# Import libraries

In [ ]:
import os
import warnings

import contextily as ctx
import geopandas as gpd
import joblib
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns

from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, Patch
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from scipy.stats import chi2
from skbio import DistanceMatrix
from skbio.stats.composition import clr
from skbio.stats.ordination import pcoa
from sklearn.covariance import GraphicalLassoCV

import mlflow

from ml.config import config
from ml.data_loading import DatabaseRSA, db_reader
from ml.features import CLRFilter, KBestFeatureSelection, MicrobiomeFeatureEngineer, ZeroColumnFilter
from ml.models import TrainTestSplit, load_and_prep_data
from ml.pipeline import build_modelling_pipeline

warnings.filterwarnings("ignore", message="invalid value encountered in subtract")

# Plotting Style

In [ ]:
# =============================================================================
# Publication plotting style
# =============================================================================

plt.style.use('default')

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 11,
    'axes.titlesize': 15,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.titlesize': 16,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'figure.dpi': 300,
    'savefig.dpi': 600
})




In [ ]:
samples = db_reader.DatabaseCreate(db="../databases/malmo.db")
rsa = DatabaseRSA(db="../databases/malmo.db", db_table="malmo_order")
df_order = rsa.merge_data(samples.get_samples(), rsa.sql_to_clean())
df_order = df_order.drop_duplicates(subset="sample_id")
df_order.head()

samples = db_reader.DatabaseCreate(db="../databases/malmo.db")
rsa = DatabaseRSA(db="../databases/malmo.db", db_table="malmo_phylum")
df_phylum = rsa.merge_data(samples.get_samples(), rsa.sql_to_clean())
df_phylum = df_phylum.drop_duplicates(subset="sample_id")
df_phylum.head()

# Number of Taxa in each lineage

In [ ]:
taxa = ["phylum","class","order","family","genus","species"]

for taxa_name in taxa:
    samples = db_reader.DatabaseCreate(db="../databases/malmo.db")
    rsa = DatabaseRSA(db="../databases/malmo.db",db_table=f"malmo_{taxa_name}")
    df = rsa.merge_data(samples.get_samples(),rsa.sql_to_clean())
    df = df.drop_duplicates(subset="sample_id")
    no_taxa = df.shape[1] - 4
    print(f"Number of {taxa_name}: {no_taxa}")

In [ ]:
# -----------------------------------------------------------------------------
# Compute data (adapt to your database structure)
# -----------------------------------------------------------------------------
taxa = ["phylum", "class", "order", "family", "genus", "species"]
n_taxa = []
mean_total_rsa = []

for taxon in taxa:
    # Load RSA data for this level
    samples = db_reader.DatabaseCreate(db="../databases/malmo.db")
    rsa = DatabaseRSA(db="../databases/malmo.db", db_table=f"malmo_{taxon}")
    df = rsa.merge_data(samples.get_samples(), rsa.sql_to_clean())
    df = df.drop_duplicates(subset="sample_id")
    
    # Number of distinct taxa (exclude sample_id, latitude, longitude)
    n = df.shape[1] - 4
    n_taxa.append(n)
    
    # Total RSA per sample (sum across all taxa for that sample)
    # Assuming columns after the first three are RSA values
    rsa_cols = df.columns[4:]
    total_rsa_per_sample = df[rsa_cols].sum(axis=1)
    # Mean across samples
    mean_total_rsa.append(total_rsa_per_sample.mean())

In [ ]:
# -----------------------------------------------------------------------------
# Create the double bar plot
# -----------------------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(8, 6))

# Bar width and positions
x = np.arange(len(taxa))
width = 0.35

# Bars: number of taxa (left y-axis)
bars1 = ax1.bar(x - width/2, n_taxa, width, label='Number of taxa', color='#2c7bb6', alpha=0.8)
ax1.set_xlabel('Taxonomic level')
ax1.set_ylabel('Number of taxa', color='#2c7bb6')
ax1.tick_params(axis='y', labelcolor='#2c7bb6')
ax1.set_xticks(x)
ax1.set_xticklabels(taxa, rotation=45, ha='right')

# Secondary y-axis: mean total RSA
ax2 = ax1.twinx()
bars2 = ax2.bar(x + width/2, mean_total_rsa, width, label='Mean total RSA (per sample)', color='#d7191c', alpha=0.8)
ax2.set_ylabel('Mean total RSA (per sample)', color='#d7191c')
ax2.tick_params(axis='y', labelcolor='#d7191c')
ax2.set_ylim(0, 1.05)  # RSA ranges 0–1

# Add a horizontal line at 0.5 (50% unclassified) for reference
ax2.axhline(y=0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.7)

# Legend (combine both axes)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
# Legend placed inside (upper right) with smaller font
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='upper center', fontsize='x-small', framealpha=0.9)

plt.subplots_adjust(bottom=0.2)

plt.title('Taxonomic classification summary', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig('/home/chandru/binp51/report/figures/taxa_summary_plot.png', dpi=300, bbox_inches='tight')
plt.show()

# RSA

In [ ]:
def plot_barplot_by_zone(df: pd.DataFrame, save_path: str = "stacked_barplot.png"):
    """
    Plot stacked barplot with for each zone.
    """
    # Filter dataframe
    df = df.drop(["sample_id", "latitude", "longitude"], axis=1)
    # Convert everything except zone to float
    df = df.astype({col: "float" for col in df.columns if col != "zone"})
    zone_df = df.groupby(by="zone").mean()

    zone_df = zone_df.loc[:, (zone_df >= 0.005).any(axis=0)]

    # Plot stacked bar plot
    # set the figure size
    plt.figure(figsize=(14, 14))
    zone_df.iloc[:, :10].plot(kind="bar", stacked=True, figsize=(14, 8))
    plt.title("RSA Values by Zone and Phylum")
    plt.xlabel("Zone")
    plt.ylabel("RSA Value")
    plt.legend(title="Phylum", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    return zone_df

In [ ]:
zone_df = plot_barplot_by_zone(df_phylum)

In [ ]:
df_phylum['Pseudomonadota'].max()

In [ ]:
zone_df

In [ ]:
data = df_phylum["Pseudomonadota"].dropna()
print(data.describe())
print(f"Percentiles: 5%={data.quantile(0.05)}, 95%={data.quantile(0.95)}")

In [ ]:
# Choose which order to plot
target_order = "Pseudomonadota"

# =============================================================================
# 1. LOAD MALMÖ BOUNDARY
# =============================================================================
def load_malmo_boundary():
    print("Fetching Malmö boundary from OpenStreetMap...")
    admin = ox.geocode_to_gdf("Malmö, Sweden")
    print(f"Boundary loaded: {admin['name'].iloc[0]}")
    return admin

boundary = load_malmo_boundary()

# =============================================================================
# 2. LOAD ADDITIONAL VECTOR LAYERS (water, parks, roads)
# =============================================================================
def load_osm_features(place, tags):
    try:
        gdf = ox.features_from_place(place, tags=tags)
        print(f"  Loaded {len(gdf)} features for tags {tags}")
        return gdf
    except Exception as e:
        print(f"  Skipping {tags}: {e}")
        return gpd.GeoDataFrame()

place_name = "Malmö, Sweden"

print("Downloading additional OSM layers...")
water = load_osm_features(place_name, {"natural": "water", "waterway": ["river", "canal"]})
parks = load_osm_features(place_name, {"leisure": "park", "landuse": "grass", "natural": "wood"})

print("Downloading street network...")
try:
    graph = ox.graph_from_place(place_name, network_type="drive")
    edges = ox.graph_to_gdfs(graph, nodes=False, edges=True)
    print(f"  Loaded {len(edges)} road segments")
except Exception as e:
    print(f"  Could not load roads: {e}")
    edges = gpd.GeoDataFrame()

# =============================================================================
# 3. PREPARE YOUR DATA POINTS
# =============================================================================
df_plot = df_phylum[df_phylum[target_order].notna()].copy()
gdf_points = gpd.GeoDataFrame(
    df_plot,
    geometry=gpd.points_from_xy(df_plot.longitude, df_plot.latitude),
    crs="EPSG:4326"
)

# Reproject everything to SWEREF99 TM (EPSG:3006)
target_crs = "EPSG:3006"
boundary = boundary.to_crs(target_crs)
if not water.empty:
    water = water.to_crs(target_crs)
if not parks.empty:
    parks = parks.to_crs(target_crs)
if not edges.empty:
    edges = edges.to_crs(target_crs)
gdf_points = gdf_points.to_crs(target_crs)

# =============================================================================
# 4. CREATE THE STATIC MAP (compact layout)
# =============================================================================
# --- Reduce figure size to be more compact (10x10) ---
fig, ax = plt.subplots(figsize=(8, 8))

# --- Set background colour ---
ax.set_facecolor("#f5f5f0")

# --- Parks (soft green) ---
if not parks.empty:
    parks.plot(ax=ax, color="#d8e8c8", edgecolor="none", alpha=0.7)

# --- Water (light blue) ---
if not water.empty:
    water.plot(ax=ax, color="#c8e0f0", edgecolor="none", alpha=0.8)

# --- Roads (thin grey lines) ---
if not edges.empty:
    major = edges[edges["highway"].isin(["motorway", "trunk", "primary", "secondary"])]
    minor = edges[~edges["highway"].isin(["motorway", "trunk", "primary", "secondary"])]
    if not minor.empty:
        minor.plot(ax=ax, color="#b0b0b0", linewidth=0.5, alpha=0.6)
    if not major.empty:
        major.plot(ax=ax, color="#707070", linewidth=1.2, alpha=0.8)

# --- Malmö boundary outline ---
boundary.plot(ax=ax, facecolor="none", edgecolor="#2c3e50", linewidth=2.5)

# --- Your data points (gradient circles) ---
# Custom colormap (light grey to red)
cmap = LinearSegmentedColormap.from_list("abundance", ["#f0f0f0", "#F85307"], N=256)

sc = ax.scatter(
    gdf_points.geometry.x,
    gdf_points.geometry.y,
    s=50,
    c=gdf_points[target_order],
    cmap=cmap,
    vmin=0.3,
    vmax=0.5,
    alpha=0.9,
    edgecolors="black",
    linewidth=0.4,
    zorder=10
)

# --- Set view limits with a SMALLER margin to reduce whitespace ---
minx, miny, maxx, maxy = boundary.total_bounds
margin = 1500   # meters (was 3000, now tighter)
ax.set_xlim(minx - margin, maxx + margin)
ax.set_ylim(miny - margin, maxy + margin)

# --- Colorbar: smaller, horizontal, with label below ---
# Use horizontal orientation so the label appears naturally below
cbar = plt.colorbar(
    sc,
    ax=ax,
    orientation='horizontal',   # horizontal bar
    shrink=0.6,                # smaller width (adjust to taste)
    pad=0.02,                  # space between map and colorbar
    aspect=30,
    fraction=0.05                  # thickness (lower = thinner)
)
# Set the label (it will appear below the colorbar automatically)
cbar.set_label('Relative sequence abundance', fontsize=10)

# --- (Optional) Customise tick values if you want ---
# For example, set ticks at specific values:
# cbar.set_ticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])
# cbar.set_ticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1.0'])

# --- Clean up axes ---
ax.set_aspect("equal")
ax.axis("off")   # removes ticks and frame

plt.title(f"Distribution of {target_order}\nin Malmö and surroundings", fontsize=16, y=0.98)

# --- Manual adjustment of figure margins to remove all whitespace ---
# left, bottom, right, top (in figure coordinates, 0 to 1)
plt.subplots_adjust(
    left=0.01,   # almost no left margin
    right=0.99,  # almost no right margin
    bottom=0.01, # almost no bottom margin
    top=0.99     # almost no top margin
)

# --- Use tight_layout with reduced padding to minimise whitespace ---
plt.tight_layout(pad=0.5)   # smaller padding around the figure

# =============================================================================
# 5. SAVE AS HIGH-RES PNG AND VECTOR PDF
# =============================================================================
plt.savefig(f"/home/chandru/binp51/report/figures/malmo_{target_order}_static_compact.png", dpi=300, bbox_inches="tight")
plt.show()

print("Map saved as PNG and PDF (compact, with horizontal colorbar).")

In [ ]:
zero_filter = ZeroColumnFilter()
X_filter = zero_filter.fit_transform(X)
clr_filter = CLRFilter()
X_clr = clr_filter.transform(X_filter)
X_clr

# Figure: Baseline Estimation (Machine Learning model)

In [ ]:
def download_stage1_results():
    client = mlflow.tracking.MlflowClient()

    # 1. Get all experiment IDs
    experiments = client.search_experiments()
    experiment_ids = [exp.experiment_id for exp in experiments]

    if not experiment_ids:
        print("No experiments found.")
        return pd.DataFrame()

    # 2. Search for runs with the stage tag across all experiments
    runs = client.search_runs(
        experiment_ids=experiment_ids,  # <- now a list
        filter_string="params.cv_strategy = 'group_kfold'"
    )

    # 3. Extract data
    data = []
    for run in runs:
        row = {
            #"run_id": run.info.run_id,
            #"experiment_id": run.info.experiment_id,
            "taxonomy_level": run.data.tags.get("taxonomy_level", ""),
            "model_type": run.data.tags.get("model_type", ""),
            "cv_strategy": run.data.params.get("cv_strategy",""),
            "mean_error_km": run.data.metrics.get("cv_mean_error_km", None),
            "median_error_km": run.data.metrics.get("cv_median_error_km", None),
            "max_error_km": run.data.metrics.get("cv_max_error_km", None),
            "experiment_stage": run.data.tags.get('mlflow.runName',None),
            "features": run.data.tags.get("fe_variant"),
            "community": run.data.params.get("fe_use_community"),
            "global_graph": run.data.params.get("fe_use_global_graph"),
            "spectral": run.data.params.get("fe_use_spectral")
            
        }
        data.append(row)

    df = pd.DataFrame(data)

    return df

# Usage
os.chdir("/home/chandru/binp51/src/ml/")
mlflow_df = download_stage1_results()

In [ ]:
mlflow_df['experiment_stage']=mlflow_df['experiment_stage'].apply(lambda x: x.split("_")[0])
mlflow_df.loc[0, "features"] = "CLR_Spectral_Global_Community"
mlflow_df.head()

In [ ]:
# 1. Prepare the data: aggregate duplicate runs

stage_1_df = mlflow_df[mlflow_df['experiment_stage']=='stage1']
df_agg = stage_1_df.groupby(['model_type', 'taxonomy_level']).agg({
    'mean_error_km': 'min',
    'median_error_km': 'min',
    'max_error_km': 'min'
}).reset_index()

# 2. Define the desired order of taxa and models
taxonomy_order = ["Phylum", "Class", "Order", "Family", "Genus", "Species"]
# Map the actual column values to title case (if needed)
df_agg['taxonomy'] = df_agg['taxonomy_level'].str.title()

# Model order (from your original list)
model_order = ["XGBoost", "RandomForest", "ElasticNet", "ExtraTreesRegressor", "RidgeRegression"]

# 3. Pivot to matrices (models x taxa)
pivot_mean = df_agg.pivot(index='model_type', columns='taxonomy', values='mean_error_km')
pivot_median = df_agg.pivot(index='model_type', columns='taxonomy', values='median_error_km')
pivot_max = df_agg.pivot(index='model_type', columns='taxonomy', values='max_error_km')

# Reindex to enforce order
pivot_mean = pivot_mean.reindex(index=model_order, columns=taxonomy_order)
pivot_median = pivot_median.reindex(index=model_order, columns=taxonomy_order)
pivot_max = pivot_max.reindex(index=model_order, columns=taxonomy_order)

# 4. Create the three‑panel figure
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

cmap = "Reds"


# Mean error
sns.heatmap(pivot_mean, ax=axes[0], cmap=cmap, vmin=3.5, vmax=6,
            annot=True, fmt=".3f", cbar=True, square=True,
            cbar_kws={"shrink": 0.8})
axes[0].set_title("A: Mean Error", fontsize=14)
axes[0].set_ylabel("Model")
axes[0].set_xlabel("")
axes[0].set_yticklabels(model_order, rotation=00)

# Median error
sns.heatmap(pivot_median, ax=axes[1], cmap=cmap, vmin=4, vmax=7.5,
            annot=True, fmt=".3f", cbar=True, square=True,
            cbar_kws={"shrink": 0.8})
axes[1].set_title("B: Median Error", fontsize=14)
axes[1].set_ylabel("")
axes[1].set_xlabel("")
axes[1].set_yticklabels([])

# Maximum error
sns.heatmap(pivot_max, ax=axes[2], cmap=cmap, vmin=5.5, vmax=10,
            annot=True, fmt=".3f", cbar=True, square=True,
            cbar_kws={"shrink": 0.8})
axes[2].set_title("C: Maximum Error", fontsize=14)
axes[2].set_ylabel("")
axes[2].set_xlabel("")
axes[2].set_yticklabels([])

plt.tight_layout()
plt.savefig("/home/chandru/binp51/report/figures/three_panel_error_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

# Table: Stage 2 Feature engineering results

In [ ]:
stage_2_df_no_clr = mlflow_df[mlflow_df['experiment_stage']=="no"]
stage_2_df_no_clr

In [ ]:
stage_2_df = mlflow_df[mlflow_df['experiment_stage']=="stage2"]
stage_2_df

# Table: Stage 3 Hyperparameter Tuning Results

In [ ]:
mlflow_df[(mlflow_df['model_type'] == 'ExtraTreesRegressor') & (mlflow_df['taxonomy_level'] == 'order')]

# Table: Wilcoxon statistic on the validation dataset for all three experiment stages

In [ ]:
# Fold errors from each stage (same order: folds 1-7)
baseline = [2.74, 3.15, 6.00, 5.20, 2.98, 4.43, 2.79]
fe_untuned = [2.72, 3.16, 6.07, 5.27, 2.82, 4.40, 2.78]
fe_tuned = [2.63, 3.15, 6.01, 5.22, 2.88, 4.50, 2.74]

def compare_stages(name_a, folds_a, name_b, folds_b):
    """Run Wilcoxon signed-rank test comparing two stages."""
    diff = np.array(folds_a) - np.array(folds_b)
    mean_diff = np.mean(diff)
    
    print(f"\n{'='*50}")
    print(f"Comparison: {name_a} vs {name_b}")
    print(f"{'='*50}")
    print(f"Mean {name_a}: {np.mean(folds_a):.4f} km")
    print(f"Mean {name_b}: {np.mean(folds_b):.4f} km")
    print(f"Average improvement: {mean_diff:.4f} km")
    
    # Wilcoxon test: alternative='greater' means folds_a > folds_b (i.e., B is better)
    stat, p_value = stats.wilcoxon(folds_a, folds_b, alternative='greater')
    print(f"Wilcoxon statistic: {stat:.4f}")
    print(f"P-value: {p_value:.4f}")
    
    if p_value < 0.05:
        print(f"✅ {name_b} is SIGNIFICANTLY better than {name_a} (p < 0.05)")
    else:
        print(f"⚠️ Not significant (p ≥ 0.05)")
    return p_value

# Run all three comparisons
print("WILCOXON SIGNED-RANK TEST RESULTS")
print("==================================")

p1 = compare_stages("Baseline", baseline, "Untuned FE", fe_untuned)
p2 = compare_stages("Baseline", baseline, "Tuned FE", fe_tuned)
p3 = compare_stages("Untuned FE", fe_untuned, "Tuned FE", fe_tuned)

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Baseline vs Untuned FE:  p = {p1:.4f}")
print(f"Baseline vs Tuned FE:    p = {p2:.4f}")
print(f"Untuned FE vs Tuned FE:  p = {p3:.4f}")

# Table: Wilcoxon test for tuned vs baseline significance test across 10 different random states (1..11)

In [ ]:
# ---- Extract paired errors for each metric ----
baseline_mean = [3.0446, 3.2923, 3.2740, 3.3262, 2.8403, 3.0371, 3.1210, 3.1017, 2.9343, 2.9769]
tuned_mean   = [2.5136, 2.8864, 2.7172, 2.8070, 2.5002, 2.6780, 2.5005, 2.7267, 2.6867, 2.5276]

baseline_median = [3.0768, 3.2217, 2.7673, 3.2775, 2.6824, 2.8391, 2.8700, 2.7698, 2.7294, 2.9183]
tuned_median   = [2.2558, 2.6627, 2.3252, 2.6627, 2.2558, 2.3998, 2.3486, 2.4997, 2.3893, 2.4469]

baseline_max = [6.8234, 7.0483, 6.8351, 7.5727, 8.0078, 6.4512, 7.1947, 7.0423, 7.3918, 7.5088]
tuned_max   = [5.4375, 6.7045, 6.2889, 6.4310, 6.4310, 6.0002, 6.0002, 6.4310, 6.4310, 6.0002]

# ---- Run Wilcoxon tests (alternative='greater' means tuned is lower) ----
w_mean, p_mean = stats.wilcoxon(baseline_mean, tuned_mean, alternative='greater')
w_median, p_median = stats.wilcoxon(baseline_median, tuned_median, alternative='greater')
w_max, p_max = stats.wilcoxon(baseline_max, tuned_max, alternative='greater')

# ---- Apply Bonferroni correction ----
alpha = 0.05
n_tests = 3
bonferroni_alpha = alpha / n_tests

print("=" * 60)
print("WILCOXON SIGNED‑RANK TESTS (Baseline vs Tuned FE)")
print("=" * 60)
print(f"Metric     | Statistic | P‑value | Significant (p < {bonferroni_alpha:.4f})?")
print("-" * 60)
print(f"Mean Error | {w_mean:.4f}    | {p_mean:.4f}  | {'✅ YES' if p_mean < bonferroni_alpha else '❌ NO'}")
print(f"Median Error| {w_median:.4f}    | {p_median:.4f}  | {'✅ YES' if p_median < bonferroni_alpha else '❌ NO'}")
print(f"Max Error  | {w_max:.4f}    | {p_max:.4f}  | {'✅ YES' if p_max < bonferroni_alpha else '❌ NO'}")
print("=" * 60)

# Summary for paper
print("\n📝 How to report this in your paper:")
print("-" * 60)
print(f"Mean improvement:   {np.mean(baseline_mean) - np.mean(tuned_mean):.4f} km (p = {p_mean:.4f})")
print(f"Median improvement: {np.mean(baseline_median) - np.mean(tuned_median):.4f} km (p = {p_median:.4f})")
print(f"Max improvement:    {np.mean(baseline_max) - np.mean(tuned_max):.4f} km (p = {p_max:.4f})")

In [ ]:
# -----------------------------------------------------------------------------
# 1. LOAD MAP & DATA (same as before)
# -----------------------------------------------------------------------------
print("Loading map and model...")
place_name = "Malmö, Sweden"
target_crs = "EPSG:3006"

boundary = ox.geocode_to_gdf(place_name).to_crs(target_crs)

def load_osm_features(place, tags):
    try:
        gdf = ox.features_from_place(place, tags=tags)
        return gdf.to_crs(target_crs) if not gdf.empty else gpd.GeoDataFrame()
    except:
        return gpd.GeoDataFrame()

water = load_osm_features(place_name, {"natural": "water", "waterway": ["river", "canal"]})
parks = load_osm_features(place_name, {"leisure": "park", "landuse": "grass", "natural": "wood"})

try:
    graph = ox.graph_from_place(place_name, network_type="drive")
    edges = ox.graph_to_gdfs(graph, nodes=False, edges=True).to_crs(target_crs)
    major = edges[edges["highway"].isin(["motorway", "trunk", "primary", "secondary"])]
    minor = edges[~edges["highway"].isin(["motorway", "trunk", "primary", "secondary"])]
except:
    major = gpd.GeoDataFrame()
    minor = gpd.GeoDataFrame()

# Load model
SEED = 37
original_rs = config.data_splitting.random_state
config.data_splitting.random_state = SEED
config.database.table = "malmo_order"
df = load_and_prep_data()
splitter = TrainTestSplit(df, n_splits=config.data_splitting.n_splits,
                          test_size=config.data_splitting.test_size)

X_test, y_test_zone, y_test_coords = splitter.get_test_data()
X_test = X_test.reindex(columns=splitter.X_cv.columns, fill_value=0.0)

pipeline = joblib.load("/home/chandru/binp51/src/ml/final_tuned_model_group_kfold.joblib")
preds = pipeline.predict(X_test)

true_coords = y_test_coords[["X_meters", "Y_meters"]].values
config.data_splitting.random_state = original_rs

# -----------------------------------------------------------------------------
# 2. CALCULATE ERRORS
# -----------------------------------------------------------------------------
errors = np.sqrt(np.sum((preds - true_coords)**2, axis=1)) / 1000.0
RADIUS = 1000  # 1 km in meters

# -----------------------------------------------------------------------------
# 3. CREATE BUFFER CIRCLES WITH SLIGHT TRANSPARENCY
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 10))
ax.set_facecolor("#f5f5f0")

# Basemap layers
if not parks.empty: parks.plot(ax=ax, color="#d8e8c8", edgecolor="none", alpha=0.7)
if not water.empty: water.plot(ax=ax, color="#c8e0f0", edgecolor="none", alpha=0.8)
if not minor.empty: minor.plot(ax=ax, color="#b0b0b0", linewidth=0.5, alpha=0.6)
if not major.empty: major.plot(ax=ax, color="#707070", linewidth=1.2, alpha=0.8)
boundary.plot(ax=ax, facecolor="none", edgecolor="#2c3e50", linewidth=2.5)

# Plot each prediction as a transparent circle with error magnitude
# Use a colormap to show error intensity
from matplotlib.colors import LinearSegmentedColormap, Normalize
cmap = LinearSegmentedColormap.from_list("error_cmap", ["#2b83ba", "#fdae61", "#d7191c"], N=256)
norm = Normalize(vmin=0, vmax=errors.max() * 1.1)

for i, (x, y) in enumerate(preds):
    # Circle with transparency based on error
    alpha = 0.08 + 0.08 * (1 - errors[i] / errors.max())  # lower error = more transparent
    color = cmap(norm(errors[i]))
    circle = plt.Circle((x, y), RADIUS, color=color, alpha=0.15, linewidth=0.8, fill=True)
    ax.add_patch(circle)
    # Draw just the border
    circle_border = plt.Circle((x, y), RADIUS, color=color, alpha=0.5, linewidth=1.0, fill=False)
    ax.add_patch(circle_border)

# Plot predicted centroids (small dots)
ax.scatter(preds[:, 0], preds[:, 1], c='red', s=30, alpha=0.7, marker='x', linewidths=1.5, label='Predicted centroid', zorder=10)

# Plot true locations (larger dots)
ax.scatter(true_coords[:, 0], true_coords[:, 1], c='#2c3e50', s=50, alpha=0.8, marker='o', edgecolors='white', linewidth=0.5, label='True location', zorder=11)

# View limits
minx, miny, maxx, maxy = boundary.total_bounds
margin = 2000
ax.set_xlim(minx - margin, maxx + margin)
ax.set_ylim(miny - margin, maxy + margin)
ax.set_aspect("equal")
ax.axis("off")

# Legend
legend_elements = [
    Patch(facecolor="#2b83ba", alpha=0.3, edgecolor="#2b83ba", label="Low error (≤ 2 km)"),
    Patch(facecolor="#fdae61", alpha=0.3, edgecolor="#fdae61", label="Medium error (2–4 km)"),
    Patch(facecolor="#d7191c", alpha=0.3, edgecolor="#d7191c", label="High error (> 4 km)"),
    Line2D([0], [0], marker="x", color="red", linestyle="None", markersize=10, label="Predicted centroid"),
    Line2D([0], [0], marker="o", color="white", markerfacecolor="#2c3e50", markersize=10, label="True location")
]
ax.legend(handles=legend_elements, loc="upper right", fontsize=9, framealpha=0.9)

ax.set_title(f"1 km Prediction Uncertainty Zones (Tuned Model, Seed = {SEED})", fontsize=14)

plt.tight_layout(pad=0.5)
plt.savefig("/home/chandru/binp51/report/figures/prediction_confidence_circles.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Best Params: {'n_estimators': 200, 'min_weight_fraction_leaf': 0.0, 'min_samples_split': 2, 'min_samples_leaf': 2, 'min_impurity_decrease': 0.0, 'max_leaf_nodes': 50, 'max_features': 'log2', 'max_depth': 5, 'bootstrap': True}

In [13]:
import joblib
from sklearn.ensemble import ExtraTreesRegressor

# 1. Load the pipeline
pipeline = joblib.load("/home/chandru/binp51/src/ml/final_tuned_model_group_kfold.joblib")

print("=" * 60)
print("PIPELINE STRUCTURE")
print("=" * 60)

# 2. Print all steps in the pipeline
print("Steps in pipeline:")
for i, (name, step) in enumerate(pipeline.steps):
    print(f"  {i}: {name} -> {step.__class__.__name__}")

# 3. Access the model step
# The model is the last step, usually named 'model'
model_step = pipeline.named_steps['model']  # or pipeline.steps[-1][1]

print(f"\nModel step type: {type(model_step)}")

# 4. If it's a MultiOutputRegressor, get the base estimator
if hasattr(model_step, 'estimators_'):
    # MultiOutputRegressor has one estimator per output
    base_estimator = model_step.estimators_[0]
    print(f"Base estimator type: {type(base_estimator)}")
else:
    base_estimator = model_step

# 5. Get all parameters of the base estimator
all_params = base_estimator.get_params()

print("\n" + "=" * 60)
print("ALL PARAMETERS")
print("=" * 60)
for key, value in all_params.items():
    print(f"  {key}: {value}")

# 6. Compare with default parameters to find tuned ones
print("\n" + "=" * 60)
print("TUNED PARAMETERS (non-default)")
print("=" * 60)

# Get default parameters for ExtraTreesRegressor
default_params = ExtraTreesRegressor().get_params()

tuned_params = {}
for key, value in all_params.items():
    if key in default_params and value != default_params[key]:
        tuned_params[key] = value
        print(f"  {key}: {value} (default: {default_params[key]})")

# 7. If you want just the tuned params as a dict for your code
print("\n" + "=" * 60)
print("TUNED PARAMETERS DICT (for use in code)")
print("=" * 60)
print(tuned_params)

PIPELINE STRUCTURE
Steps in pipeline:
  0: prevalence -> ZeroColumnFilter
  1: clr -> CLRFilter
  2: network_features -> GraphLaplacianFeatureEngineer
  3: model -> MultiOutputRegressor

Model step type: <class 'sklearn.multioutput.MultiOutputRegressor'>
Base estimator type: <class 'sklearn.ensemble._forest.ExtraTreesRegressor'>

ALL PARAMETERS
  bootstrap: True
  ccp_alpha: 0.0
  criterion: squared_error
  max_depth: 5
  max_features: log2
  max_leaf_nodes: 50
  max_samples: None
  min_impurity_decrease: 0.0
  min_samples_leaf: 2
  min_samples_split: 2
  min_weight_fraction_leaf: 0.0
  monotonic_cst: None
  n_estimators: 200
  n_jobs: None
  oob_score: False
  random_state: 42
  verbose: 0
  warm_start: False

TUNED PARAMETERS (non-default)
  bootstrap: True (default: False)
  max_depth: 5 (default: None)
  max_features: log2 (default: 1.0)
  max_leaf_nodes: 50 (default: None)
  min_samples_leaf: 2 (default: 1)
  n_estimators: 200 (default: 100)
  random_state: 42 (default: None)

TUN